# Repository execution setup
Use Python 3.10/3.11. For QLoRA select a CUDA GPU before installing. This notebook calls repository scripts; training code lives in `src/llm_grading/`.
Do not upload private student data to a public notebook. Keep outputs private and use persistent storage if the runtime may disconnect.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json
REPO = Path.cwd() if (Path.cwd() / "scripts/train.py").exists() else Path("/content/LLM-APA" if Path("/content").exists() else "/kaggle/working/LLM-APA")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/duykhang2103/LLM-APA.git", str(REPO)], check=True)
os.chdir(REPO)
# Notebook kernels may use another Python version; repository scripts use Python 3.11.
subprocess.run([sys.executable, "-m", "pip", "install", "uv", "pyyaml"], check=True)
notebook_env = REPO / ".notebook-venv"
setup_env = dict(os.environ, UV_PROJECT_ENVIRONMENT=str(notebook_env))
subprocess.run([sys.executable, "-m", "uv", "sync", "--python", "3.11", "--group", "dev", "--group", "ml"], cwd=REPO, env=setup_env, check=True)
PYTHON = str(notebook_env / ("Scripts/python.exe" if os.name == "nt" else "bin/python"))
def run(script, *args):
    subprocess.run([PYTHON, str(REPO / "scripts" / script), *map(str, args)], cwd=REPO, check=True)


## Optional persistent storage
Set `PERSISTENT` below to a mounted volume, Google Drive, or a Kaggle output directory. On Colab you may mount Drive with the next cell. This is optional; no cloud SDK is used by the training code. Kaggle `/kaggle/working` must be exported/saved before ending the session.


In [ ]:
# Optional Colab only; uncomment if needed.
# from google.colab import drive
# drive.mount("/content/drive")
PERSISTENT = Path("/content/drive/MyDrive/LLM-APA-runs") if Path("/content/drive/MyDrive").exists() else REPO / "outputs/notebook-runs"
PERSISTENT.mkdir(parents=True, exist_ok=True)


# Qwen3.5-4B QLoRA: local / Colab / Kaggle
Select a CUDA GPU. Local/cloud execution uses the same code. A tiny sample pilot does not prove that the full token budget fits your GPU. Use the smoke gate before spending training time.


In [ ]:
import yaml
TASK = "task1"  # task1 / task2 / task3
DATA_ROOT = Path("/content/private-data")  # change to supplied teacher-format directory
TRAIN_FILE = None   # optional official train file/directory
VAL_FILE = None     # official dev file/directory; set BOTH train and val when supplied
TEST_FILE = None    # lecturer test file/directory, if supplied
VERSION = "F0-v1"  # use F0-v0-smoke, F0-v2, etc. for separate runs
RESUME_FROM = None  # e.g. str(PERSISTENT / "task1/F0-v1/checkpoint-100")
config = json.loads(json.dumps(yaml.safe_load((REPO / f"configs/{TASK}/qwen_lora.yaml").read_text())))
config["experiment"].update(id=f"{TASK}-{VERSION}", notes="first QLoRA baseline")
config["data"].update(input_path=str(DATA_ROOT), train_path=TRAIN_FILE, validation_path=VAL_FILE, test_path=TEST_FILE,
                      prepared_dir=str(PERSISTENT / TASK / "prepared-v1"))
config["saving"]["output_dir"] = str(PERSISTENT / TASK / VERSION)
# Adjust max_sequence_length to measured GPU capacity; code never silently truncates evidence.
CONFIG = PERSISTENT / f"{TASK}-{VERSION}-config.json"
CONFIG.write_text(json.dumps(config, indent=2))
run("prepare_data.py", "--config", CONFIG, "--training")
PREPARED = Path(config["data"]["prepared_dir"]) / ("config-" + config["experiment"]["id"] + ".json")


## Smoke gate
Checks model/tokenizer, finite loss, LoRA parameter updates, checkpoint save and adapter reload, one prediction/schema/evaluator. Inspect artifacts and failures. This creates a separate `smoke/` directory; full training starts fresh.


In [ ]:
run("smoke.py", "--config", PREPARED, "--training")


## Train or resume
A Trainer checkpoint includes optimizer/scheduler state. An `adapter/` folder supports inference or a fresh warm start, not exact resume. A changed dataset/template/adapter contract requires a new run. Set `training.warm_start_adapter` for a deliberate new experiment from old weights.


In [ ]:
args = ["--config", PREPARED]
if RESUME_FROM:
    args += ["--resume-from-checkpoint", RESUME_FROM]
run("train.py", *args)


## Validation inference and metrics
Use the saved exact base revision. Prediction errors receive one schema repair retry; no heuristic fallback. Inspect per-exam/level/conflict slices as well as overall scores.


In [ ]:
OUT = Path(config["saving"]["output_dir"])
manifest = json.loads((OUT / "run_manifest.json").read_text())
inference = json.loads(PREPARED.read_text())
inference["model"].update(revision=manifest["model_revision"], adapter_path=manifest["adapter_path"])
INFERENCE = OUT / "inference_config.json"
INFERENCE.write_text(json.dumps(inference, indent=2))
run("predict.py", "--config", INFERENCE, "--split", "val")
run("validate_predictions.py", "--task", TASK, "--input", OUT / "predictions.json")
run("evaluate.py", "--config", INFERENCE, "--split", "val", "--predictions", OUT / "predictions.json")
json.loads((OUT / "metrics.json").read_text())


## Export / lecturer acceptance
The run directory is already persistent if configured above. Export adapters, logs and configs privately. Final acceptance requires only the input path and frozen inference config; do not tune/retrain on private test targets. Packaging retains the repository prediction contract; confirm the lecturer's final external format if it differs.


In [ ]:
# Optional private export (may be large if Trainer checkpoints are included):
# import shutil
# shutil.make_archive(str(PERSISTENT / f"{TASK}-{VERSION}-artifacts"), "zip", OUT)
# run("predict.py", "--config", INFERENCE, "--input", TEST_FILE, "--output", OUT / "final_predictions.json")
# run("package_predictions.py", "--task", TASK, "--input", OUT / "final_predictions.json", "--output", OUT / "submission.zip")
